# BGE Reranker 파인튜닝 — tax-rag

**소요 시간:** T4 GPU 기준 30분 이내  
**비용:** $0.5 미만 (무료 T4 사용 가능)

## 사전 준비
1. 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
2. 왼쪽 열쇠 아이콘(Secrets) → `GITHUB_TOKEN` 등록 (read 권한)

---

## 0. GPU 확인

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout[:800])
else:
    print('GPU 없음. 런타임 > 런타임 유형 변경 > T4 GPU 선택 후 재실행')

## 1. 의존성 설치

In [ ]:
!pip install -q sentence-transformers==3.3.1 torch transformers accelerate
print('설치 완료')

## 2. GitHub Token 설정

In [ ]:
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN') or ''
except Exception:
    GITHUB_TOKEN = ''

# 직접 입력 필요 시 주석 해제
# GITHUB_TOKEN = 'ghp_xxxxx'

if not GITHUB_TOKEN:
    raise RuntimeError('GITHUB_TOKEN 미설정. Secrets에 등록하거나 위 주석을 해제하세요.')
print('GITHUB_TOKEN: 설정됨')

## 3. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive')
WORK_DIR   = Path('/content/tax-rag')
print('Drive 마운트 완료')

## 4. 레포 Clone

In [ ]:
import subprocess, sys, shutil

REPO = 'Raw-Agent/korean-tax-rag'

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)

result = subprocess.run(
    ['git', 'clone', '--depth', '1', f'https://{GITHUB_TOKEN}@github.com/{REPO}.git', str(WORK_DIR)],
    capture_output=True, text=True
)
if result.returncode != 0:
    raise RuntimeError(f'clone 실패:\n{result.stderr[:800]}')

sys.path.insert(0, str(WORK_DIR))

LOCAL_PAIRS_PATH = WORK_DIR / 'data' / 'reranker_pairs.jsonl'
lines = LOCAL_PAIRS_PATH.read_text(encoding='utf-8').strip().splitlines()
print(f'clone 완료 — reranker_pairs.jsonl: {len(lines):,}줄')

## 5. 데이터 검증

In [ ]:
import json

complete, pos_only = [], []
for line in LOCAL_PAIRS_PATH.read_text(encoding='utf-8').splitlines():
    line = line.strip()
    if not line:
        continue
    try:
        rec = json.loads(line)
        if rec.get('positive') and rec.get('negative'):
            complete.append(rec)
        elif rec.get('positive'):
            pos_only.append(rec)
    except Exception:
        pass

print(f'complete triplet : {len(complete):,}건')
print(f'positive-only    : {len(pos_only):,}건')
print(f'소스 수           : {len(set(r["source"] for r in complete))}개')

## 6. 파인튜닝 실행

In [ ]:
import random
from sentence_transformers import CrossEncoder, InputExample
from torch.utils.data import DataLoader

BASE_MODEL   = 'BAAI/bge-reranker-v2-m3'
OUTPUT_DIR   = WORK_DIR / 'models' / 'bge-reranker-tax-rag'
EPOCHS       = 4
BATCH_SIZE   = 8    # VRAM 부족 시 4로 낮추세요
LR           = 1e-5
MAX_LENGTH   = 512
WARMUP_RATIO = 0.05

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def split_by_source(data, eval_ratio=0.15, seed=42):
    grouped = {}
    for rec in data:
        grouped.setdefault(rec.get('source', '?'), []).append(rec)
    sources = list(grouped)
    rng = random.Random(seed)
    rng.shuffle(sources)
    n_eval = max(1, min(int(len(sources) * eval_ratio), len(sources) - 1))
    eval_src = set(sources[:n_eval])
    return (
        [r for r in data if r.get('source') not in eval_src],
        [r for r in data if r.get('source') in eval_src],
    )

train_data, eval_data = split_by_source(complete)

train_samples = []
for rec in train_data:
    train_samples.append(InputExample(texts=[rec['query'], rec['positive']], label=1.0))
    train_samples.append(InputExample(texts=[rec['query'], rec['negative']], label=0.0))

print(f'학습 triplet : {len(train_data):,}건')
print(f'평가 triplet : {len(eval_data):,}건')
print(f'\n모델 로드 중: {BASE_MODEL}')

model = CrossEncoder(BASE_MODEL, num_labels=1, max_length=MAX_LENGTH)
train_dl = DataLoader(train_samples, shuffle=True, batch_size=BATCH_SIZE)
warmup_steps = int(len(train_dl) * EPOCHS * WARMUP_RATIO)

print(f'파인튜닝 시작 (epochs={EPOCHS}, batch={BATCH_SIZE}, lr={LR}, warmup={warmup_steps}steps)\n')
model.fit(
    train_dataloader=train_dl,
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    optimizer_params={'lr': LR},
    output_path=str(OUTPUT_DIR),
    show_progress_bar=True,
)
print(f'\n파인튜닝 완료: {OUTPUT_DIR}')

## 7. 평가

In [ ]:
import numpy as np

eval_model = CrossEncoder(str(OUTPUT_DIR), max_length=MAX_LENGTH)

pairs  = [(rec['query'], rec['positive']) for rec in complete] + \
         [(rec['query'], rec['negative']) for rec in complete]
labels = [1] * len(complete) + [0] * len(complete)

scores = eval_model.predict(pairs, show_progress_bar=True)
preds  = (scores > 0.5).astype(int)
acc    = (preds == np.array(labels)).mean()

print(f'\n전체 정확도: {acc:.4f} (목표: 0.85 이상)')
print('목표 달성 — 모델 프로모션 가능' if acc >= 0.85 else f'목표 미달 ({acc:.4f} < 0.85)')

## 8. Google Drive에 저장

In [ ]:
import shutil

DRIVE_MODEL_DIR = DRIVE_ROOT / 'tax-rag' / 'bge-reranker-tax-rag'
if DRIVE_MODEL_DIR.exists():
    shutil.rmtree(DRIVE_MODEL_DIR)

shutil.copytree(OUTPUT_DIR, DRIVE_MODEL_DIR)
print(f'Drive 저장 완료: {DRIVE_MODEL_DIR}')

for f in sorted(DRIVE_MODEL_DIR.iterdir()):
    print(f'  {f.name:40s} {f.stat().st_size/1024/1024:6.1f} MB')

## 9. 로컬 프로모션

Drive에서 `bge-reranker-tax-rag` 폴더 다운로드 후:

**1. 모델 폴더 교체 (PowerShell)**
```powershell
Move-Item -Force "C:\Users\next0\Downloads\bge-reranker-tax-rag" `
          "C:\Users\next0\claude-test\tax-rag\data\models\bge-reranker-tax-rag"
```

**2. `.env` 수정**
```
BGE_RERANKER_MODEL=data/models/bge-reranker-tax-rag
```

**3. 베이스라인 eval 실행**
```bash
python -m scripts.run_baseline_eval --workers 3
```

> 정확도 85% 이상 시 `.env` 자동 업데이트. 미달 시 기존 모델 유지.

---
**재파인튜닝 트리거:** `data/red_wins/` 50건 초과 시 이 노트북 재실행.